# 04 · Feature extraction

| | |
|---|---|
| **入力** | `data/clean/<subject>.csv` |
| **出力** | 窓化した特徴量行列 + フィット済み `StandardScaler`（`.npz` / `.joblib`） |

流れ: EEG の帯域分割 → 時系列ブロック分割 → スライディング窓 → 特徴量 ブランチ → 標準化.

| ブランチ | 窓長 | 特徴量 |
|---|---|---|
| EEG Hjorth | 0.5 s | チャネルごとの activity + mobility, 帯域ごと（α, β） |
| EMG RMS | 0.2 s | 筋ごとの二乗平均平方根 |
| モーション | 0.2 s | マーカー平均位置 |
| 環境 | – | 椅子高, 段差高 |

In [ ]:
import sys
from pathlib import Path

# notebooks/ から実行したときに motion_intent パッケージを import 可能にする
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from motion_intent import config

In [ ]:
import joblib
from sklearn.preprocessing import StandardScaler

from motion_intent.preprocessing import (
    merge_xz_all, drop_single_axis_marker_cols, append_band_columns,
    bandpass_by_session,
)
from motion_intent.windowing import (
    split_by_time_block, make_reference_indices, windows_at,
)
from motion_intent.features import (
    extract_hjorth_fast, extract_rms, extract_mean_position, build_env_features,
)

## Load & band-split

In [ ]:
subject = 'subjectA'
df = pd.read_csv(config.CLEAN_DIR / f'{subject}.csv')
df = drop_single_axis_marker_cols(merge_xz_all(df))
df = df.dropna().reset_index(drop=True)

# 必要ならワンホット列から整数ラベルを作る
if 'label' not in df and set(config.CLASS_NAMES).issubset(df.columns):
    df['label'] = df[config.CLASS_NAMES].to_numpy().argmax(axis=1)

# 運動野中心 EEG チャネルの α / β 帯域コピーを追加
df = append_band_columns(df, config.EEG_CH_FEATURE, 'Session', config.FS,
                         bands=config.EEG_BANDS, drop_original=False)
# EMG 帯域通過
df = bandpass_by_session(df, config.EMG_COLS, 'Session', config.FS,
                         band=config.EMG_BAND)

## Chronological split, per session

In [ ]:
splits = {'train': [], 'val': [], 'test': []}
for _, df_sess in df.groupby('Session'):
    tr, va, te = split_by_time_block(df_sess)
    splits['train'].append(tr); splits['val'].append(va); splits['test'].append(te)

## Windowing

In [ ]:
COLS = {
    'eeg_a':   [f'{c}_alpha' for c in config.EEG_CH_FEATURE],
    'eeg_b':   [f'{c}_beta'  for c in config.EEG_CH_FEATURE],
    'emg':     config.EMG_COLS,
    'motion':  config.MOTION_COLS,
}

def window_split(frames):
    out = {k: [] for k in COLS}
    ys = []
    for df_sess in frames:
        t = df_sess['t_sec'].to_numpy()
        ref = make_reference_indices(t, config.WIN_SEC_EEG, config.STEP_SEC)
        ref = ref[ref >= config.WIN_EEG]
        y = df_sess['label'].to_numpy()
        # 0.5 s EEG 窓の中心サンプルのラベルを採用
        ys.append(y[ref - config.WIN_EEG // 2])
        for name, cols in COLS.items():
            win = config.WIN_EEG if name.startswith('eeg') else config.WIN_EMG
            out[name].append(windows_at(df_sess[cols].to_numpy(), ref, win))
    return {k: np.concatenate(v) for k, v in out.items()}, np.concatenate(ys)

Xw, yw = {}, {}
for part in ('train', 'val', 'test'):
    Xw[part], yw[part] = window_split(splits[part])

## Feature branches

EEG 特徴量は α / β 帯域それぞれの Hjorth（activity・mobility）のみ.

In [ ]:
def features_for(part):
    X = Xw[part]
    hjorth = np.concatenate([extract_hjorth_fast(X['eeg_a']),
                             extract_hjorth_fast(X['eeg_b'])], axis=1)
    rms = extract_rms(X['emg'])
    motion = extract_mean_position(X['motion'])
    env = build_env_features(len(rms), config.DEFAULT_CHAIR_HEIGHT_M,
                             config.DEFAULT_STAIR_HEIGHT_M)
    return dict(hjorth=hjorth, rms=rms, motion=motion, env=env)

F = {part: features_for(part) for part in ('train', 'val', 'test')}

## Standardise (fit on train) and save

In [ ]:
scalers = {}
for key in ('hjorth', 'rms', 'motion'):
    sc = StandardScaler().fit(F['train'][key])
    scalers[key] = sc
    for part in ('train', 'val', 'test'):
        F[part][key] = sc.transform(F[part][key])

art_dir = config.DATA_DIR / 'features'
art_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(scalers, art_dir / f'{subject}_scalers.joblib')
for part in ('train', 'val', 'test'):
    np.savez(art_dir / f'{subject}_{part}.npz', y=yw[part], **F[part])
print('saved to', art_dir)